# F1 All Laps starter

Load the live [Formula One (F1) - All Laps & Telemetry](https://www.kaggle.com/datasets/stevefza/f1-2026-bahrain-testing-day-2) dataset, profile laps, and plot one car's 0.5s telemetry.

One row = one lap. The `telemetry` column is a JSON array of samples (`t`, `speed`, `throttle`, `brake`, `gear`, `rpm`, `drs`, `x`, `y`, `z`).

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

DATASET = "stevefza/f1-2026-bahrain-testing-day-2"
kaggle_dir = Path("/kaggle/input/f1-2026-bahrain-testing-day-2")

if kaggle_dir.exists():
    root = kaggle_dir
else:
    import kagglehub
    root = Path(kagglehub.dataset_download(DATASET))

csvs = sorted(root.rglob("All Laps*.csv"))
assert csvs, f"No All Laps CSV under {root}"
csv_path = csvs[-1]
print("Using", csv_path)

laps = pd.read_csv(csv_path, low_memory=False)
print(f"{len(laps):,} laps, {laps['event_name'].nunique()} events")
laps.head()

In [ ]:
summary = (
    laps.groupby(["event_name", "session_type"], dropna=False)
    .agg(laps=("lap_number", "size"), drivers=("driver", "nunique"))
    .reset_index()
    .sort_values("laps", ascending=False)
)
summary.head(15)

In [ ]:
timed = laps.dropna(subset=["lap_duration"]).copy()
timed = timed[timed["lap_duration"].between(60, 200)]
best = (
    timed.sort_values("lap_duration")
    .groupby(["event_name", "driver"], as_index=False)
    .first()[["event_name", "driver", "session_type", "lap_time", "lap_duration", "compound"]]
)
best.sort_values("lap_duration").head(10)

In [ ]:
sample = timed.dropna(subset=["telemetry"]).iloc[0]
points = pd.DataFrame(json.loads(sample["telemetry"]))
print(
    sample["event_name"], sample["session_type"],
    sample["driver"], "lap", sample["lap_number"],
    sample["lap_time"], f"({len(points)} samples)"
)

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(points["t"], points["speed"], color="#e10600", linewidth=1.4)
ax.set_xlabel("Lap time (s)")
ax.set_ylabel("Speed")
ax.set_title(f"{sample['driver']} — {sample['event_name']} lap {int(sample['lap_number'])}")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
points.head()